# DeiT-S + Distillation Token on ImageNet-1k (PSC)

Trains **DeiT-S** from scratch on **ImageNet-1k** with the paper's **distillation token**
(DeiT⚗, Figure 2) and hard distillation (Eq. 3). PSC / Bridges-2 only.

The student is `deit_small_distilled_patch16_224` (dim 384, depth 12, 6 heads, ~22M) with
random init. The teacher is the paper's **RegNetY-16GF**, `regnety_160.deit_in1k` — already
1000-way, so no class slicing.

| `distillation_type` | Objective |
|---|---|
| `'hard'` (default) | \(\tfrac{1}{2} L_{CE}(\psi(Z_s), y) + \tfrac{1}{2} L_{CE}(\psi(Z_s), y_t)\) |
| `'soft'` | \((1-\lambda) L_{CE} + \lambda \tau^2 \mathrm{KL}\) with \(\lambda=0.1\), \(\tau=3\) |
| `'none'` | class-token CE only (token is turned off) |

## PSC setup

Request a GPU node, then start Jupyter from `/jet/home`:

```bash
interact -p GPU-shared --gres=gpu:v100-32:1 -t 8:00:00 -A <allocation>
# or a batch job for the real 300-epoch run
module load anaconda3
conda activate <env>
export PYTHONNOUSERSITE=1
cd /jet/home/$USER
jupyter notebook --no-browser --ip=0.0.0.0
```

Checkpoints go on `/jet/home` (they survive the job). Put ImageNet on node-local `$LOCAL`
if you can; reading 1.28M JPEGs from the shared filesystem is slow.

# Paths

`DATA_ROOT` must contain `train/` and `val/`, each with 1000 WNID folders (standard ImageFolder).

In [ ]:
import os
from pathlib import Path

# ============================ FILL THESE IN ============================
DATA_ROOT = ''                              # e.g. '/local/imagenet' or the ILSVRC CLS-LOC root
WORK_DIR  = os.path.expanduser('~/deit_s_in1k_distill')
# ======================================================================

assert DATA_ROOT, "Set DATA_ROOT to the ImageNet-1k folder that contains train/ and val/"
DATA_ROOT = os.path.expanduser(DATA_ROOT)
TRAIN_DIR = os.path.join(DATA_ROOT, 'train')
VAL_DIR   = os.path.join(DATA_ROOT, 'val')
assert os.path.isdir(TRAIN_DIR), f"missing train split: {TRAIN_DIR}"
assert os.path.isdir(VAL_DIR),   f"missing val split: {VAL_DIR}"

os.makedirs(WORK_DIR, exist_ok=True)
os.environ.setdefault('HF_HOME', os.path.expanduser('~/.cache/huggingface'))
os.environ.setdefault('TORCH_HOME', os.path.expanduser('~/.cache/torch'))

print("Data root :", DATA_ROOT)
print("Train     :", TRAIN_DIR)
print("Val       :", VAL_DIR)
print("Work dir  :", WORK_DIR)
print("Scratch   :", os.environ.get('LOCAL', '(unset)'))

# GPU and packages

In [ ]:
!nvidia-smi

In [ ]:
import sys, subprocess, importlib.util

def pip_install(packages):
    if packages:
        print("installing:", " ".join(packages))
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=False)

def is_installed(pkg):
    try:
        return importlib.util.find_spec(pkg.replace('-', '_')) is not None
    except (ImportError, ValueError):
        return False

need = [p for p in ('timm', 'wandb', 'huggingface_hub') if not is_installed(p)]
pip_install(need)
# timm's pretrained hub is picky about huggingface_hub 1.x
try:
    import huggingface_hub as _hf
    if tuple(int(x) for x in _hf.__version__.split('.')[:2]) >= (1, 0):
        pip_install(['huggingface_hub<1.0'])
except Exception:
    pass

import torch, torchvision, timm
print("torch      :", torch.__version__)
print("torchvision:", torchvision.__version__)
print("timm       :", timm.__version__)
print("CUDA       :", torch.cuda.is_available(), "|", torch.cuda.device_count(), "device(s)")
for i in range(torch.cuda.device_count()):
    print("  ", i, torch.cuda.get_device_name(i))

# Imports

In [ ]:
import os, gc, json, math, time, copy, random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid
import torchvision.transforms as T
from torchvision.transforms import InterpolationMode

import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", DEVICE)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


torch.backends.cudnn.benchmark = True
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

# Config

Paper defaults: hard distillation + distillation token, 300 epochs, cosine + 5-epoch warmup,
MixUp 0.8 / CutMix 1.0, RandAugment, repeated aug, EMA 0.99996.

Linear LR scale: `lr = 5e-4 * eff_batch / 512`. The paper used batch 1024 (`lr = 1e-3`).
A single V100-32 with the RegNetY-16GF teacher usually fits **batch 64**. Raise
`batch_size` / `grad_accum_steps` if you have more memory.

In [ ]:
config = {
    'data_root'   : DATA_ROOT,
    'train_dir'   : TRAIN_DIR,
    'val_dir'     : VAL_DIR,
    'num_classes' : 1000,
    'image_size'  : 224,
    'num_workers' : min(os.cpu_count() or 4, 8),

    'auto_augment'       : 'rand-m9-mstd0.5-inc1',
    'train_interpolation': 'bicubic',
    'rrc_scale'          : (0.08, 1.0),
    're_prob'            : 0.25,
    're_mode'            : 'pixel',
    're_count'           : 1,
    'eval_crop_ratio'    : 0.875,
    'repeated_aug'       : True,
    'ra_repeats'         : 3,

    'mixup_alpha'             : 0.8,
    'cutmix_alpha'            : 1.0,
    'mix_prob'                : 1.0,
    'mixup_cutmix_switch_prob': 0.5,
    'mixup_mode'              : 'batch',
    'label_smoothing'         : 0.1,

    'distillation_type': 'hard',    # 'none' | 'soft' | 'hard'
    'distill_token'    : True,
    'distill_alpha'    : None,      # None -> paper default
    'distill_tau'      : None,
    'soft_kd_reduction': 'numel',
    'head_fusion'      : 'logit_mean',

    'teacher_model'    : 'regnety_160.deit_in1k',
    'teacher_amp'      : True,

    'model_source'   : 'timm',      # 'timm' | 'scratch'
    'patch_size'     : 16,
    'embed_dim'      : 384,
    'depth'          : 12,
    'num_heads'      : 6,
    'mlp_ratio'      : 4.0,
    'qkv_bias'       : True,
    'drop_rate'      : 0.0,
    'attn_drop_rate' : 0.0,
    'drop_path_rate' : 0.1,

    'epochs'          : 300,
    'warmup_epochs'   : 5,
    'batch_size'      : 64,         # raise on A100 / multi-GPU
    'grad_accum_steps': 4,          # effective 256; paper used 1024
    'base_lr'         : 5e-4,
    'base_batch'      : 512,
    'warmup_lr'       : 1e-6,
    'min_lr'          : 1e-5,
    'weight_decay'    : 0.05,
    'grad_clip'       : None,
    'ema'             : True,
    'ema_decay'       : 0.99996,    # paper value (ImageNet-1k, large step count)

    'amp'               : True,
    'seed'              : 42,
    'max_runtime_hours' : 47.0,     # leave headroom on a 48 h GPU job; drop to 7.5 for interact
    'checkpoint_dir'    : os.path.join(WORK_DIR, 'checkpoints'),
    'resume_from'       : None,
    'smoke_test'        : False,

    'use_wandb'     : False,
    'wandb_project' : 'imagenet1k-deit-s-distill',
    'run_name'      : 'deit_s_distill_in1k_psc',
}

config['eff_batch']  = config['batch_size'] * config['grad_accum_steps']
config['lr']         = config['base_lr'] * config['eff_batch'] / config['base_batch']
config['val_resize'] = int(round(config['image_size'] / config['eval_crop_ratio']))

assert config['batch_size'] % 2 == 0, "timm Mixup requires an even batch size"
assert config['distillation_type'] in ('none', 'soft', 'hard')

if config['distillation_type'] == 'none':
    if config['distill_token']:
        print("[note] distillation_type='none' -> distill_token=False")
    config['distill_token'] = False
    config['distill_alpha'], config['distill_tau'] = 0.0, 1.0
elif config['distillation_type'] == 'hard':
    config['distill_alpha'] = 0.5 if config['distill_alpha'] is None else config['distill_alpha']
    config['distill_tau']   = 1.0 if config['distill_tau'] is None else config['distill_tau']
else:
    config['distill_alpha'] = 0.1 if config['distill_alpha'] is None else config['distill_alpha']
    config['distill_tau']   = 3.0 if config['distill_tau'] is None else config['distill_tau']

if config['smoke_test']:
    config.update(epochs=2, warmup_epochs=1, ema=False)

seed_everything(config['seed'])
os.makedirs(config['checkpoint_dir'], exist_ok=True)

print(json.dumps({k: str(v) for k, v in config.items()}, indent=2))
print(f"\nEffective batch: {config['eff_batch']}  ->  scaled LR: {config['lr']:.3e}")
print(f"Distillation   : {config['distillation_type']} | token: {config['distill_token']} | "
      f"alpha: {config['distill_alpha']} | tau: {config['distill_tau']}")

# Transforms

In [ ]:
from timm.data import create_transform


def create_transforms(cfg, is_training):
    if is_training:
        return create_transform(
            input_size=cfg['image_size'],
            is_training=True,
            scale=cfg['rrc_scale'],
            auto_augment=cfg['auto_augment'],
            interpolation=cfg['train_interpolation'],
            re_prob=cfg['re_prob'],
            re_mode=cfg['re_mode'],
            re_count=cfg['re_count'],
            mean=IMAGENET_MEAN,
            std=IMAGENET_STD,
        )
    return T.Compose([
        T.Resize(cfg['val_resize'], interpolation=InterpolationMode.BICUBIC),
        T.CenterCrop(cfg['image_size']),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


train_transforms = create_transforms(config, is_training=True)
val_transforms   = create_transforms(config, is_training=False)
print("TRAIN:", train_transforms)
print("\nVAL  :", val_transforms)

# Datasets

Standard `ImageFolder`. Class folders must be ImageNet WNIDs (`n01440764`, …); sorted order
is the official 1000-way index, which matches the teacher.

In [ ]:
class RASampler(torch.utils.data.Sampler):
    def __init__(self, dataset, num_repeats=3, shuffle=True, seed=0):
        if num_repeats < 1:
            raise ValueError("num_repeats should be greater than 0")
        self.num_samples = len(dataset)
        self.num_repeats = num_repeats
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0
        self.num_selected_samples = (self.num_samples // 256) * 256 or self.num_samples

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __iter__(self):
        if self.shuffle:
            g = torch.Generator()
            g.manual_seed(self.seed + self.epoch)
            indices = torch.randperm(self.num_samples, generator=g)
        else:
            indices = torch.arange(self.num_samples)
        indices = torch.repeat_interleave(indices, repeats=self.num_repeats, dim=0)
        return iter(indices[:self.num_selected_samples].tolist())

    def __len__(self):
        return self.num_selected_samples


print("Indexing ImageNet (first run can take a few minutes)...")
train_dataset = ImageFolder(config['train_dir'], transform=train_transforms)
val_dataset   = ImageFolder(config['val_dir'],   transform=val_transforms)

classes = train_dataset.classes
class_to_idx = train_dataset.class_to_idx
assert classes == val_dataset.classes, "train/val class folders do not match"
assert len(classes) == config['num_classes'], \
    f"found {len(classes)} classes, expected {config['num_classes']}"
assert classes == sorted(classes), "class folders are not sorted WNIDs — teacher labels will not align"

try:
    from timm.data import ImageNetInfo
    info = ImageNetInfo('imagenet-1k')
    in1k = list(info.label_names())
    if classes == in1k:
        idx_to_name = {i: info.index_to_description(i) for i in range(len(classes))}
        print("Class order matches timm ImageNet-1k.")
    else:
        idx_to_name = {i: w for i, w in enumerate(classes)}
        print("[warn] class order differs from timm ImageNet-1k — teacher transfer will be wrong.")
except Exception as e:
    idx_to_name = {i: w for i, w in enumerate(classes)}
    print("[note] could not load ImageNetInfo:", e)

ra_sampler = RASampler(train_dataset, num_repeats=config['ra_repeats']) \
    if config['repeated_aug'] else None

train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    sampler=ra_sampler,
    shuffle=(ra_sampler is None),
    num_workers=config['num_workers'],
    pin_memory=True,
    drop_last=True,
    persistent_workers=config['num_workers'] > 0,
    prefetch_factor=4 if config['num_workers'] > 0 else None,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config['batch_size'] * 2,
    shuffle=False,
    num_workers=config['num_workers'],
    pin_memory=True,
    persistent_workers=config['num_workers'] > 0,
)

print(f"Train images : {len(train_dataset):,}  ({len(train_loader):,} batches)")
print(f"Val images   : {len(val_dataset):,}  ({len(val_loader):,} batches)")
print(f"Classes      : {len(classes)}  e.g. {classes[:4]}")
if ra_sampler is not None:
    print(f"RASampler    : {config['ra_repeats']}x -> {len(ra_sampler):,} samples/epoch")

## MixUp / CutMix

In [ ]:
from timm.data import Mixup

mixup_fn = Mixup(
    mixup_alpha=config['mixup_alpha'],
    cutmix_alpha=config['cutmix_alpha'],
    cutmix_minmax=None,
    prob=config['mix_prob'],
    switch_prob=config['mixup_cutmix_switch_prob'],
    mode=config['mixup_mode'],
    label_smoothing=config['label_smoothing'],
    num_classes=config['num_classes'],
)
mixup_active = config['mixup_alpha'] > 0 or config['cutmix_alpha'] > 0


def apply_batch_mix(images, labels):
    if not mixup_active:
        return images, labels
    return mixup_fn(images, labels)


print("Mixup active:", mixup_active)

In [ ]:
def denormalize(x):
    mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
    return (x.cpu() * std + mean).clamp(0, 1)

images, labels = next(iter(val_loader))
grid = make_grid(denormalize(images[:16]), nrow=8, padding=2)
fig, ax = plt.subplots(figsize=(12, 3.4))
ax.imshow(grid.permute(1, 2, 0).numpy())
ax.axis('off')
ax.set_title(" | ".join(str(idx_to_name[int(l)])[:18] for l in labels[:8]), fontsize=8)
plt.tight_layout(); plt.show()

# Student (DeiT-S + distillation token)

In [ ]:
def trunc_normal_(tensor, std=0.02):
    return nn.init.trunc_normal_(tensor, mean=0.0, std=std, a=-2 * std, b=2 * std)


class DropPath(nn.Module):
    def __init__(self, p: float = 0.0):
        super().__init__()
        self.p = p

    def forward(self, x):
        if self.p == 0.0 or not self.training:
            return x
        keep = 1.0 - self.p
        mask = x.new_empty((x.shape[0],) + (1,) * (x.ndim - 1)).bernoulli_(keep)
        return x * mask / keep


class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=384):
        super().__init__()
        assert img_size % patch_size == 0
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        B, C, H, W = x.shape
        assert H == self.img_size and W == self.img_size
        return self.proj(x).flatten(2).transpose(1, 2)


class Attention(nn.Module):
    def __init__(self, dim, num_heads=6, qkv_bias=True, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.attn_drop_p = attn_drop
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        self._fused = hasattr(F, 'scaled_dot_product_attention')

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        if self._fused:
            x = F.scaled_dot_product_attention(
                q, k, v, dropout_p=self.attn_drop_p if self.training else 0.0
            )
        else:
            attn = self.attn_drop(((q @ k.transpose(-2, -1)) * self.scale).softmax(-1))
            x = attn @ v
        return self.proj_drop(self.proj(x.transpose(1, 2).reshape(B, N, C)))


class Mlp(nn.Module):
    def __init__(self, dim, hidden_dim, drop=0.0):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))


class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, qkv_bias=True,
                 drop=0.0, attn_drop=0.0, drop_path=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = Attention(dim, num_heads, qkv_bias, attn_drop, drop)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = Mlp(dim, int(dim * mlp_ratio), drop)
        self.drop_path = DropPath(drop_path)

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class DeiT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, num_classes=1000,
                 embed_dim=384, depth=12, num_heads=6, mlp_ratio=4.0, qkv_bias=True,
                 drop_rate=0.0, attn_drop_rate=0.0, drop_path_rate=0.1, distilled=False):
        super().__init__()
        self.num_classes = num_classes
        self.embed_dim = embed_dim
        self.distilled = distilled
        self.num_tokens = 2 if distilled else 1
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.dist_token = nn.Parameter(torch.zeros(1, 1, embed_dim)) if distilled else None
        self.pos_embed = nn.Parameter(torch.zeros(1, self.patch_embed.num_patches + self.num_tokens, embed_dim))
        self.pos_drop = nn.Dropout(drop_rate)
        dpr = torch.linspace(0, drop_path_rate, depth).tolist()
        self.blocks = nn.ModuleList([
            Block(embed_dim, num_heads, mlp_ratio, qkv_bias, drop_rate, attn_drop_rate, dpr[i])
            for i in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim, eps=1e-6)
        self.head = nn.Linear(embed_dim, num_classes)
        self.head_dist = nn.Linear(embed_dim, num_classes) if distilled else None
        self.apply(self._init_weights)
        trunc_normal_(self.pos_embed, std=0.02)
        trunc_normal_(self.cls_token, std=0.02)
        if self.dist_token is not None:
            trunc_normal_(self.dist_token, std=0.02)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def no_weight_decay(self):
        return {'pos_embed', 'cls_token', 'dist_token'}

    def forward_features(self, x):
        x = self.patch_embed(x)
        tokens = [self.cls_token.expand(x.shape[0], -1, -1)]
        if self.dist_token is not None:
            tokens.append(self.dist_token.expand(x.shape[0], -1, -1))
        x = self.pos_drop(torch.cat(tokens + [x], dim=1) + self.pos_embed)
        for blk in self.blocks:
            x = blk(x)
        return self.norm(x)

    def forward(self, x):
        x = self.forward_features(x)
        cls = x[:, 0]
        out = {'feats': cls}
        if self.distilled:
            logits_cls = self.head(cls)
            logits_dist = self.head_dist(x[:, 1])
            out['logits_cls'] = logits_cls
            out['logits_dist'] = logits_dist
            out['out'] = (logits_cls + logits_dist) / 2
        else:
            out['out'] = self.head(cls)
        return out


def deit_small_patch16(num_classes=1000, img_size=224, **kw):
    defaults = dict(patch_size=16, embed_dim=384, depth=12, num_heads=6, mlp_ratio=4.0)
    defaults.update(kw)
    return DeiT(img_size=img_size, num_classes=num_classes, **defaults)


def fuse_logits(outputs):
    if isinstance(outputs, dict):
        return outputs['out']
    if isinstance(outputs, (tuple, list)):
        cls_logits, dist_logits = outputs
        if config['head_fusion'] == 'softmax_sum':
            return (cls_logits.softmax(1) + dist_logits.softmax(1)).log()
        return (cls_logits + dist_logits) / 2
    return outputs


def student_forward(model, images):
    out = model(images)
    if isinstance(out, dict):
        if out.get('logits_dist') is not None:
            return out['logits_cls'], out['logits_dist']
        return out['out']
    return out


def forward_logits(model, images):
    return fuse_logits(model(images))


def unwrap(model):
    return model.module if isinstance(model, nn.DataParallel) else model

In [ ]:
STUDENT_NAME = ('deit_small_distilled_patch16_224' if config['distill_token']
                else 'deit_small_patch16_224')

if config['model_source'] == 'timm':
    model = timm.create_model(
        STUDENT_NAME,
        pretrained=False,
        num_classes=config['num_classes'],
        img_size=config['image_size'],
        drop_rate=config['drop_rate'],
        drop_path_rate=config['drop_path_rate'],
    )
    if config['distill_token']:
        model.set_distilled_training(True)
else:
    model = deit_small_patch16(
        num_classes=config['num_classes'],
        img_size=config['image_size'],
        patch_size=config['patch_size'],
        embed_dim=config['embed_dim'],
        depth=config['depth'],
        num_heads=config['num_heads'],
        mlp_ratio=config['mlp_ratio'],
        qkv_bias=config['qkv_bias'],
        drop_rate=config['drop_rate'],
        attn_drop_rate=config['attn_drop_rate'],
        drop_path_rate=config['drop_path_rate'],
        distilled=config['distill_token'],
    )

model = model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Student      : {STUDENT_NAME} ({config['model_source']})")
print(f"Distill token: {config['distill_token']}")
print(f"Params       : {n_params:,} ({n_params / 1e6:.2f} M)")

dummy = torch.randn(2, 3, config['image_size'], config['image_size'], device=DEVICE)
model.train()
with torch.no_grad():
    train_out = student_forward(model, dummy)
if isinstance(train_out, tuple):
    print(f"Train : class {tuple(train_out[0].shape)} + distill {tuple(train_out[1].shape)}")
else:
    print(f"Train : {tuple(train_out.shape)}")
assert config['distill_token'] == isinstance(train_out, tuple)

model.eval()
with torch.no_grad():
    print(f"Eval  : {tuple(forward_logits(model, dummy).shape)}")

# Metrics

In [ ]:
class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = self.avg = self.sum = 0.0
        self.count = 0

    def update(self, val, n: int = 1):
        self.val = float(val)
        self.sum += float(val) * n
        self.count += n
        self.avg = self.sum / max(self.count, 1)


def topk_accuracy(logits, targets, topk=(1, 5)):
    if targets.ndim == 2:
        targets = targets.argmax(dim=1)
    maxk = min(max(topk), logits.size(1))
    _, preds = logits.topk(maxk, dim=1, largest=True, sorted=True)
    correct = preds.t().eq(targets.view(1, -1))
    out = []
    for k in topk:
        k = min(k, maxk)
        out.append(correct[:k].reshape(-1).float().sum(0) * 100.0 / targets.size(0))
    return out


def format_time(seconds):
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h:d}h{m:02d}m{s:02d}s"

# Teacher

`regnety_160.deit_in1k` is already 1000-way. First call downloads the weights into
`~/.cache` on `/jet/home` (needs outbound network on the compute node, once).

In [ ]:
teacher = None
t_top1 = t_top5 = None

if config['distillation_type'] != 'none':
    teacher_model = timm.create_model(
        config['teacher_model'],
        pretrained=True,
        num_classes=1000,
    )
    teacher_model = teacher_model.to(DEVICE).eval()
    for p in teacher_model.parameters():
        p.requires_grad_(False)

    class TeacherWrapper(nn.Module):
        def __init__(self, net):
            super().__init__()
            self.net = net

        @torch.no_grad()
        def forward(self, x):
            return self.net(x)

    teacher = TeacherWrapper(teacher_model).to(DEVICE).eval()
    n_t = sum(p.numel() for p in teacher_model.parameters())
    print(f"Teacher: {config['teacher_model']}  {n_t / 1e6:.1f}M  frozen 1000-way")

    with torch.no_grad():
        probe = torch.randn(2, 3, config['image_size'], config['image_size'], device=DEVICE)
        out = teacher(probe)
        print("Teacher output:", tuple(out.shape))
        assert out.shape == (2, 1000)
else:
    print("distillation_type='none' -> no teacher")

In [ ]:
@torch.no_grad()
def evaluate_teacher(teacher, dataloader, device, amp=True, max_batches=40):
    top1, top5 = AverageMeter(), AverageMeter()
    n = min(max_batches, len(dataloader))
    for i, (images, labels) in enumerate(tqdm(dataloader, total=n, desc="Teacher probe", leave=False)):
        if i >= n:
            break
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=amp and device == 'cuda'):
            logits = teacher(images)
        a1, a5 = topk_accuracy(logits.float(), labels, topk=(1, 5))
        top1.update(a1.item(), images.size(0))
        top5.update(a5.item(), images.size(0))
    return top1.avg, top5.avg


if teacher is not None:
    t_top1, t_top5 = evaluate_teacher(teacher, val_loader, DEVICE, amp=config['teacher_amp'])
    print(f"Teacher probe (~{40 * config['batch_size'] * 2} val images): "
          f"top-1 {t_top1:.2f}% | top-5 {t_top5:.2f}%")
    if t_top1 < 70:
        print("[WARN] teacher probe is low — check that val folders are official WNIDs "
              "in the same sorted order as train.")

# Distillation loss

In [ ]:
class DistillationLoss(nn.Module):
    def __init__(self, base_criterion, teacher_model, distillation_type, alpha, tau,
                 soft_reduction='numel'):
        super().__init__()
        assert distillation_type in ('none', 'soft', 'hard')
        self.base_criterion = base_criterion
        self.teacher_model = teacher_model
        self.distillation_type = distillation_type
        self.alpha = alpha
        self.tau = tau
        self.soft_reduction = soft_reduction

    def forward(self, inputs, outputs, labels):
        outputs_kd = None
        if not isinstance(outputs, torch.Tensor):
            outputs, outputs_kd = outputs
        base_loss = self.base_criterion(outputs, labels)
        if self.distillation_type == 'none':
            z = torch.zeros((), device=base_loss.device)
            return base_loss, {'base': base_loss.detach(), 'distill': z}
        if outputs_kd is None:
            outputs_kd = outputs
        with torch.no_grad():
            teacher_outputs = self.teacher_model(inputs)
        if self.distillation_type == 'soft':
            T = self.tau
            kl = F.kl_div(
                F.log_softmax(outputs_kd / T, dim=1),
                F.log_softmax(teacher_outputs / T, dim=1),
                reduction='sum', log_target=True,
            ) * (T * T)
            denom = outputs_kd.numel() if self.soft_reduction == 'numel' else outputs_kd.size(0)
            distillation_loss = kl / denom
        else:
            distillation_loss = F.cross_entropy(outputs_kd, teacher_outputs.argmax(dim=1))
        loss = base_loss * (1 - self.alpha) + distillation_loss * self.alpha
        return loss, {'base': base_loss.detach(), 'distill': distillation_loss.detach()}

# Optimizer and schedule

In [ ]:
from timm.loss import SoftTargetCrossEntropy, LabelSmoothingCrossEntropy
from timm.scheduler import create_scheduler_v2

base_criterion = SoftTargetCrossEntropy() if mixup_active \
    else LabelSmoothingCrossEntropy(smoothing=config['label_smoothing'])
criterion = DistillationLoss(
    base_criterion, teacher,
    config['distillation_type'], config['distill_alpha'], config['distill_tau'],
    soft_reduction=config['soft_kd_reduction'],
)
val_criterion = nn.CrossEntropyLoss()

base_model = unwrap(model)
skip_keywords = set(base_model.no_weight_decay()) if hasattr(base_model, 'no_weight_decay') else set()
decay, no_decay = [], []
for name, p in base_model.named_parameters():
    if not p.requires_grad:
        continue
    if p.ndim <= 1 or name.endswith('.bias') or any(k in name for k in skip_keywords):
        no_decay.append(p)
    else:
        decay.append(p)

optimizer = torch.optim.AdamW(
    [{'params': decay, 'weight_decay': config['weight_decay']},
     {'params': no_decay, 'weight_decay': 0.0}],
    lr=config['lr'], betas=(0.9, 0.999), eps=1e-8,
)

scheduler, _ = create_scheduler_v2(
    optimizer, sched='cosine', num_epochs=config['epochs'],
    min_lr=config['min_lr'], warmup_lr=config['warmup_lr'],
    warmup_epochs=config['warmup_epochs'], cooldown_epochs=0,
)

amp_on = config['amp'] and DEVICE == 'cuda'
try:
    scaler = torch.amp.GradScaler('cuda', enabled=amp_on)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=amp_on)

print("criterion:", type(base_criterion).__name__, "+", config['distillation_type'])
print(f"decay / no-decay tensors: {len(decay)} / {len(no_decay)}")
print(f"schedule: {config['epochs']} epochs, {config['warmup_epochs']} warmup, lr {config['lr']:.3e}")

In [ ]:
class ModelEma:
    def __init__(self, model, decay=0.99996, warmup=True):
        self.module = copy.deepcopy(unwrap(model)).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay = decay
        self.warmup = warmup
        self.updates = 0

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        d = min(self.decay, (1 + self.updates) / (10 + self.updates)) if self.warmup else self.decay
        msd = unwrap(model).state_dict()
        for k, v in self.module.state_dict().items():
            src = msd[k].detach()
            if v.dtype.is_floating_point:
                v.mul_(d).add_(src.to(v.device), alpha=1.0 - d)
            else:
                v.copy_(src.to(v.device))

    def state_dict(self):
        return self.module.state_dict()

    def load_state_dict(self, sd):
        self.module.load_state_dict(sd)


model_ema = ModelEma(model, decay=config['ema_decay']) if config['ema'] else None
print("EMA:", "on" if model_ema else "off")

# Train / val

In [ ]:
def train_epoch(model, dataloader, optimizer, scaler, device, criterion,
                ema=None, accum_steps=1, grad_clip=None, epoch=0, limit_batches=None):
    model.train()
    loss_meter, top1_meter, top5_meter = AverageMeter(), AverageMeter(), AverageMeter()
    base_meter, kd_meter = AverageMeter(), AverageMeter()
    n_batches = len(dataloader) if limit_batches is None else min(limit_batches, len(dataloader))
    progress = tqdm(dataloader, total=n_batches, desc=f"Train {epoch + 1}",
                    dynamic_ncols=True, leave=False)
    optimizer.zero_grad(set_to_none=True)
    amp_enabled = scaler.is_enabled()

    for i, (images, labels) in enumerate(progress):
        if i >= n_batches:
            break
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        hard_labels = labels
        images, labels = apply_batch_mix(images, labels)

        with torch.amp.autocast('cuda', enabled=amp_enabled):
            outputs = student_forward(model, images)
            loss, parts = criterion(images, outputs, labels)
            logits = fuse_logits(outputs)

        scaler.scale(loss / accum_steps).backward()
        is_step = ((i + 1) % accum_steps == 0) or (i + 1 == n_batches)
        if is_step:
            if grad_clip is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            if ema is not None:
                ema.update(model)

        if math.isfinite(loss.item()):
            acc1, acc5 = topk_accuracy(logits.detach().float(), hard_labels, topk=(1, 5))
            bs = images.size(0)
            loss_meter.update(loss.item(), bs)
            top1_meter.update(acc1.item(), bs)
            top5_meter.update(acc5.item(), bs)
            base_meter.update(parts['base'].item(), bs)
            kd_meter.update(parts['distill'].item(), bs)
        progress.set_postfix(loss=f"{loss_meter.avg:.4f}", ce=f"{base_meter.avg:.3f}",
                             kd=f"{kd_meter.avg:.3f}", top1=f"{top1_meter.avg:.2f}%")

    return top1_meter.avg, top5_meter.avg, loss_meter.avg, base_meter.avg, kd_meter.avg


@torch.no_grad()
def validate(model, dataloader, device, criterion, amp=True, desc="Val", return_preds=False):
    model.eval()
    loss_meter, top1_meter, top5_meter = AverageMeter(), AverageMeter(), AverageMeter()
    all_preds, all_targets = [], []
    for images, labels in tqdm(dataloader, desc=desc, dynamic_ncols=True, leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=amp and device == 'cuda'):
            logits = fuse_logits(model(images)).float()
        loss = criterion(logits, labels)
        acc1, acc5 = topk_accuracy(logits, labels, topk=(1, 5))
        bs = images.size(0)
        loss_meter.update(loss.item(), bs)
        top1_meter.update(acc1.item(), bs)
        top5_meter.update(acc5.item(), bs)
        if return_preds:
            all_preds.append(logits.argmax(1).cpu())
            all_targets.append(labels.cpu())
    if return_preds:
        return (top1_meter.avg, top5_meter.avg, loss_meter.avg,
                torch.cat(all_preds).numpy(), torch.cat(all_targets).numpy())
    return top1_meter.avg, top5_meter.avg, loss_meter.avg

# WandB (optional)

In [ ]:
WANDB_KEY = ""

run = None
if config['use_wandb'] and WANDB_KEY:
    import wandb
    wandb.login(key=WANDB_KEY)
    run = wandb.init(project=config['wandb_project'], name=config['run_name'], config=config)
    print("WandB:", run.url)
else:
    print("WandB off")

# Checkpoints

`last.pth` is written every epoch under `WORK_DIR`. Re-run the notebook to resume.
Do not load the scheduler state — `step(start_epoch)` rebuilds the cosine from the epoch.

In [ ]:
CKPT_DIR = config['checkpoint_dir']
os.makedirs(CKPT_DIR, exist_ok=True)


def save_checkpoint(path, model, optimizer, scheduler, scaler, ema, epoch, metrics, history):
    torch.save({
        'model_state_dict'    : unwrap(model).state_dict(),
        'ema_state_dict'      : ema.state_dict() if ema is not None else None,
        'optimizer_state_dict': optimizer.state_dict() if optimizer is not None else None,
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'scaler_state_dict'   : scaler.state_dict() if scaler is not None else None,
        'epoch'               : epoch,
        'metrics'             : metrics,
        'history'             : history,
        'config'              : config,
        'classes'             : classes,
    }, path)


def load_checkpoint(path, model, optimizer=None, scaler=None, ema=None, device='cuda'):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    unwrap(model).load_state_dict(ckpt['model_state_dict'])
    if ema is not None and ckpt.get('ema_state_dict') is not None:
        ema.load_state_dict(ckpt['ema_state_dict'])
    if optimizer is not None and ckpt.get('optimizer_state_dict'):
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if scaler is not None and ckpt.get('scaler_state_dict'):
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    print(f"Loaded {path} | epoch {ckpt.get('epoch', -1) + 1} | {ckpt.get('metrics')}")
    return ckpt


start_epoch = 0
best_top1 = 0.0
history = {'epoch': [], 'lr': [], 'train_loss': [], 'train_top1': [],
           'train_ce': [], 'train_kd': [],
           'val_loss': [], 'val_top1': [], 'val_top5': [], 'ema_top1': []}

resume_path = config['resume_from'] or os.path.join(CKPT_DIR, 'last.pth')
if resume_path and os.path.exists(resume_path):
    ckpt = load_checkpoint(resume_path, model, optimizer, scaler, model_ema, DEVICE)
    start_epoch = ckpt.get('epoch', -1) + 1
    best_top1 = max(ckpt.get('metrics', {}).get('best_top1', 0.0), 0.0)
    loaded = ckpt.get('history') or {}
    if set(loaded) >= set(history):
        history = {k: list(loaded[k]) for k in history}
    scheduler.step(start_epoch)
    print(f"Resume epoch {start_epoch + 1}/{config['epochs']}  best {best_top1:.2f}%  "
          f"lr {optimizer.param_groups[0]['lr']:.3e}")
else:
    print("Training from scratch.")

# Training

In [ ]:
if 'run_start_time' not in globals():
    run_start_time = time.time()

limit_batches = 20 if config['smoke_test'] else None
stop_reason = "completed"

for epoch in range(start_epoch, config['epochs']):
    print(f"\n=== Epoch {epoch + 1}/{config['epochs']} ===")
    epoch_start = time.time()
    curr_lr = optimizer.param_groups[0]['lr']
    if ra_sampler is not None:
        ra_sampler.set_epoch(epoch)

    train_top1, train_top5, train_loss, train_ce, train_kd = train_epoch(
        model, train_loader, optimizer, scaler, DEVICE, criterion,
        ema=model_ema, accum_steps=config['grad_accum_steps'],
        grad_clip=config['grad_clip'], epoch=epoch, limit_batches=limit_batches,
    )
    print(f"Train | top1 {train_top1:6.2f}% | loss {train_loss:.4f} "
          f"(ce {train_ce:.3f} | kd {train_kd:.3f}) | lr {curr_lr:.3e}")

    val_top1, val_top5, val_loss = validate(
        model, val_loader, DEVICE, val_criterion, amp=config['amp'], desc=f"Val {epoch + 1}"
    )
    print(f"Val   | top1 {val_top1:6.2f}% | top5 {val_top5:6.2f}% | loss {val_loss:.4f}")

    ema_top1 = float('nan')
    if model_ema is not None:
        ema_top1, ema_top5, ema_loss = validate(
            model_ema.module, val_loader, DEVICE, val_criterion, amp=config['amp'], desc="Val EMA"
        )
        print(f"EMA   | top1 {ema_top1:6.2f}% | top5 {ema_top5:6.2f}%")

    score = max(val_top1, ema_top1 if model_ema is not None and not math.isnan(ema_top1) else 0.0)
    for key, value in [('epoch', epoch + 1), ('lr', curr_lr),
                       ('train_ce', train_ce), ('train_kd', train_kd),
                       ('train_loss', train_loss), ('train_top1', train_top1),
                       ('val_loss', val_loss), ('val_top1', val_top1),
                       ('val_top5', val_top5), ('ema_top1', ema_top1)]:
        history[key].append(value)

    metrics = {'train_loss': train_loss, 'train_ce': train_ce, 'train_kd': train_kd,
               'train_top1': train_top1, 'val_loss': val_loss,
               'val_top1': val_top1, 'val_top5': val_top5, 'ema_top1': ema_top1,
               'lr': curr_lr, 'best_top1': max(best_top1, score)}
    if run is not None:
        run.log(metrics, step=epoch + 1)

    with open(os.path.join(CKPT_DIR, 'history.json'), 'w') as f:
        json.dump(history, f, indent=2)

    last_path = os.path.join(CKPT_DIR, 'last.pth')
    save_checkpoint(last_path, model, optimizer, scheduler, scaler, model_ema, epoch, metrics, history)
    if run is not None:
        wandb.save(last_path)

    if score > best_top1:
        best_top1 = score
        best_path = os.path.join(CKPT_DIR, 'best.pth')
        save_checkpoint(best_path, model, optimizer, scheduler, scaler, model_ema, epoch, metrics, history)
        if run is not None:
            wandb.save(best_path)
        print(f"  -> new best top-1: {best_top1:.2f}%")

    scheduler.step(epoch + 1)
    elapsed = time.time() - run_start_time
    print(f"  epoch {format_time(time.time() - epoch_start)} | "
          f"session {format_time(elapsed)} | best {best_top1:.2f}%")
    if elapsed > config['max_runtime_hours'] * 3600:
        stop_reason = "time budget reached"
        print(f"\nStopping: {stop_reason}. Resume from {last_path}")
        break

print(f"\nDone ({stop_reason}). Best val top-1: {best_top1:.2f}%")
if run is not None:
    run.finish()

# Results

In [ ]:
hist = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
axes[0].plot(hist['epoch'], hist['train_loss'], label='train')
axes[0].plot(hist['epoch'], hist['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].plot(hist['epoch'], hist['train_top1'], label='train')
axes[1].plot(hist['epoch'], hist['val_top1'], label='val')
if hist['ema_top1'].notna().any():
    axes[1].plot(hist['epoch'], hist['ema_top1'], label='EMA', ls='--')
axes[1].set_title('Top-1'); axes[1].legend(); axes[1].grid(alpha=.3)
axes[2].plot(hist['epoch'], hist['lr'])
axes[2].set_title('LR'); axes[2].grid(alpha=.3)
plt.tight_layout(); plt.show()
hist.tail(8)

# Final eval and Table 3 heads

In [ ]:
best_path = os.path.join(CKPT_DIR, 'best.pth')
if os.path.exists(best_path):
    ckpt = load_checkpoint(best_path, model, device=DEVICE)
    m = ckpt.get('metrics', {})
    if ckpt.get('ema_state_dict') is not None and (m.get('ema_top1') or 0) > (m.get('val_top1') or 0):
        unwrap(model).load_state_dict(ckpt['ema_state_dict'])
        print("Using EMA weights.")
else:
    print("best.pth missing — evaluating in-memory weights.")

val_top1, val_top5, val_loss, preds, targets = validate(
    model, val_loader, DEVICE, val_criterion, amp=config['amp'],
    desc="Final val", return_preds=True,
)
print(f"Final | top-1 {val_top1:.2f}% | top-5 {val_top5:.2f}% | loss {val_loss:.4f}")

In [ ]:
@torch.no_grad()
def head_logits(model, images):
    net = unwrap(model)
    out = net(images)
    if isinstance(out, dict):
        return out.get('logits_cls', out['out']), out.get('logits_dist')
    if hasattr(net, 'head_dist') and net.head_dist is not None:
        feats = net.forward_features(images)
        cls_tok, dist_tok = feats[:, 0], feats[:, 1]
        for extra in ('fc_norm', 'head_drop'):
            if hasattr(net, extra):
                cls_tok = getattr(net, extra)(cls_tok)
                dist_tok = getattr(net, extra)(dist_tok)
        return net.head(cls_tok), net.head_dist(dist_tok)
    return fuse_logits(out), None


@torch.no_grad()
def validate_heads(model, dataloader, device, amp=True):
    model.eval()
    meters = {k: (AverageMeter(), AverageMeter()) for k in ('class', 'distillation', 'fused')}
    for images, labels in tqdm(dataloader, desc="Heads", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=amp and device == 'cuda'):
            logits_cls, logits_dist = head_logits(model, images)
        candidates = {'class': logits_cls.float()}
        if logits_dist is not None:
            candidates['distillation'] = logits_dist.float()
            candidates['fused'] = fuse_logits((candidates['class'], candidates['distillation']))
        for name, logits in candidates.items():
            a1, a5 = topk_accuracy(logits, labels, topk=(1, 5))
            meters[name][0].update(a1.item(), images.size(0))
            meters[name][1].update(a5.item(), images.size(0))
    return {n: (m1.avg, m5.avg) for n, (m1, m5) in meters.items() if m1.count}


head_results = validate_heads(model, val_loader, DEVICE, amp=config['amp'])
print(f"{'classifier':<16}{'top-1':>8}{'top-5':>8}")
for name, (a1, a5) in head_results.items():
    print(f"{name:<16}{a1:>8.2f}{a5:>8.2f}")
if t_top1 is not None:
    print(f"{'teacher-probe':<16}{t_top1:>8.2f}{t_top5:>8.2f}")

In [ ]:
OUT_DIR = WORK_DIR
hist.to_csv(f"{OUT_DIR}/history.csv", index=False)
with open(f"{OUT_DIR}/history.json", 'w') as f:
    json.dump(history, f, indent=2)
with open(f"{OUT_DIR}/head_results.json", 'w') as f:
    json.dump({'distillation_type': config['distillation_type'],
               'distill_token': config['distill_token'],
               'heads': head_results,
               'teacher_probe_top1': t_top1}, f, indent=2)

torch.save({
    'model_state_dict': unwrap(model).state_dict(),
    'config': config,
    'classes': classes,
    'val_top1': val_top1,
    'val_top5': val_top5,
    'head_results': head_results,
}, f"{OUT_DIR}/deit_s_imagenet1k_distill.pth")

print("wrote", OUT_DIR)

# Notes

- Fill in `DATA_ROOT` (must have `train/` + `val/` with 1000 WNID folders) and run on a GPU node.
- Teacher weights download once into `~/.cache`. The compute node needs outbound HTTPS the first time.
- Paper batch is 1024. This notebook defaults to 64×4 = 256 so a V100-32 can hold DeiT-S plus RegNetY-16GF. Scale `batch_size` / `grad_accum_steps` if you have more GPU.
- Expect ~79–81% top-1 for DeiT-S with hard distillation + token after 300 epochs (paper Table 2/5).